<h3>PDF 실습 준비</h3>
<p><strong>개념</strong> 디지털 PDF는 텍스트 층을 가질 수 있지만 읽기 순서와 표 구조는 별도로 확인해야 합니다. 스캔 PDF는 이미지이므로 텍스트 추출 결과가 비어 있을 수 있습니다.</p>

In [1]:
# %pip은 현재 노트북 커널에 PDF 읽기 패키지를 설치합니다.
# -q는 설치 로그를 줄이고 ==는 실습에서 검증한 버전을 고정합니다.
%pip install -q "pypdf==6.18.0" "pdfplumber==0.11.10"


Note: you may need to restart the kernel to use updated packages.


<h3>실행 위치와 데이터 폴더 확인</h3>
<p>프로젝트 루트의 <code>labs/data/rag-corpus</code>를 찾습니다. 일반 PDF와 스캔 PDF가 모두 있는지 확인한 다음 파싱을 시작합니다. 이 절에는 API 키가 필요하지 않습니다.</p>

In [2]:
# 두 라이브러리의 클래스와 현재 경로를 다루는 Path를 가져옵니다.
# 상위 폴더를 순서대로 확인하므로 저장소 내부의 어느 폴더에서도 실행할 수 있습니다.
from pathlib import Path
from pypdf import PdfReader
import pdfplumber

repo = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "labs/data/rag-corpus").is_dir():
        repo = candidate
        break
if repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 실행하세요.")
pdf_path = repo / "labs/data/rag-corpus/pdf/easy/E02.pdf"
scan_path = repo / "labs/data/rag-corpus/pdf/scanned/S01-E01-scan.pdf"
for path in (pdf_path, scan_path):
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.relative_to(repo), path.stat().st_size, "bytes")


labs/data/rag-corpus/pdf/easy/E02.pdf 306440 bytes
labs/data/rag-corpus/pdf/scanned/S01-E01-scan.pdf 919812 bytes


<h3>PDF 객체 열기</h3>
<p><code>PdfReader</code>는 PDF 전체를 나타내는 객체를 반환합니다. <code>pages</code> 속성에는 쪽 객체가 순서대로 들어 있습니다.</p>

In [3]:
# 파일 경로를 PdfReader에 전달해 PDF 객체를 만듭니다.
# len(reader.pages)는 실제 쪽 객체의 개수를 반환합니다.
reader = PdfReader(pdf_path)
print("객체:", type(reader).__name__)
print("전체 쪽 수:", len(reader.pages))


객체: PdfReader
전체 쪽 수: 5


<h3>기본 텍스트 관찰</h3>
<p>목록 위치는 0부터 시작하므로 <code>pages[1]</code>은 두 번째 쪽입니다. <code>extract_text()</code>가 값을 반환하지 않으면 <code>or ""</code>로 빈 문자열을 사용합니다.</p>

In [4]:
# 두 번째 쪽 객체에서 일반 텍스트를 추출합니다.
# 길이와 앞부분을 함께 출력해 값이 있는지와 읽기 순서를 눈으로 확인합니다.
page_number = 2
page = reader.pages[page_number - 1]
plain = page.extract_text() or ""
print("선택한 쪽:", page_number)
print("글자 수:", len(plain))
print("본문 앞부분:", plain[:120].replace("\n", " "))


선택한 쪽: 2
글자 수: 884
본문 앞부분: 환영합니다 별빛제과에 오신 것을 환영합니다. 별빛제과는 1987년 강릉의 작은 빵집에서 시작해 지금은 전국 3개 공장과 220개 매장을 둔 제과 회사입니다. 대표 제품은 '별빛 카스텔라'와 '바다소금 크래커'이며, 


<h3>레이아웃과 표 결과 비교</h3>
<p><code>pdfplumber</code>로 같은 쪽을 읽습니다. 레이아웃 문자열과 표 목록은 서로 다른 반환값이므로 길이와 표 개수를 각각 확인합니다.</p>

In [5]:
# with 블록은 PDF를 사용한 뒤 자동으로 닫습니다.
# extract_tables()는 찾은 표마다 행과 열을 담은 중첩 목록을 반환합니다.
with pdfplumber.open(pdf_path) as pdf:
    selected = pdf.pages[page_number - 1]
    layout = selected.extract_text(layout=True) or ""
    tables = selected.extract_tables()
print("기본 글자 수:", len(plain))
print("레이아웃 글자 수:", len(layout))
print("표 개수:", len(tables))


기본 글자 수: 884
레이아웃 글자 수: 5311
표 개수: 1


<h3>선택한 쪽을 바꾸어 비교</h3>
<p>파서 설정은 그대로 두고 선택값만 2쪽에서 1쪽으로 바꿉니다. 두 글자 수가 다르면 같은 파일 안에서도 쪽마다 결과가 달라진다는 뜻입니다.</p>

In [6]:
# 함수는 쪽 번호를 받아 해당 쪽의 텍스트 길이를 반환합니다.
# page_number - 1은 사람이 보는 번호를 0부터 시작하는 목록 위치로 바꿉니다.
def text_length(reader: PdfReader, page_number: int) -> int:
    text = reader.pages[page_number - 1].extract_text() or ""
    return len(text)

page_1_chars = text_length(reader, 1)
page_2_chars = text_length(reader, 2)
print("1쪽 글자 수:", page_1_chars)
print("2쪽 글자 수:", page_2_chars)


1쪽 글자 수: 58
2쪽 글자 수: 884


<h3>일부러 실패해 보기</h3>
<p><strong>Before 원인</strong> 스캔 PDF도 디지털 PDF처럼 텍스트가 바로 나온다고 가정하면 빈 문자열을 정상 본문으로 처리하게 됩니다.</p>

In [7]:
# 스캔 PDF의 모든 쪽에 텍스트가 있다고 가정해 실패를 재현합니다.
# 반복문은 각 쪽의 추출 결과를 extracted 목록에 순서대로 저장합니다.
try:
    scan = PdfReader(scan_path)
    extracted = []
    for page in scan.pages:
        extracted.append(page.extract_text() or "")
    has_text_on_every_page = all(text.strip() for text in extracted)
    assert has_text_on_every_page, "텍스트 층이 없는 쪽이 있습니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: 텍스트 층이 없는 쪽이 있습니다


<p><strong>After 수정</strong> 텍스트 길이가 20자 미만인 쪽을 OCR 후보로 분류합니다. 이 코드는 OCR을 실행하거나 인식 성공을 주장하지 않습니다.</p>

In [8]:
# enumerate(..., 1)은 쪽 번호를 1부터 만들고, 짧은 텍스트 쪽만 기록합니다.
# 이 픽스처의 모든 쪽이 후보인지 확인하되 OCR 자체는 실행하지 않습니다.
scan = PdfReader(scan_path)
missing = []
for page_number, page in enumerate(scan.pages, 1):
    text = page.extract_text() or ""
    if len(text.strip()) < 20:
        missing.append(page_number)
expected_pages = list(range(1, len(scan.pages) + 1))
assert missing == expected_pages
print("OCR candidate pages:", missing)


OCR candidate pages: [1, 2, 3, 4, 5]
